# WID3002 Natural Language Processing - Exam Preparation & Cache Warmup

This notebook compiles all the libraries, tokenisers, corpora, and models used throughout the NLP tutorials, along with extra models of the same family that might appear in the finals.

Run the cells below sequentially to pre-install and pre-cache all assets so you do not have to wait for large downloads during the exam.

⚠️ **Library Version Compatibility Nuances by Python Version:**
* **Python 3.14+ (Edge cases / Pre-release)**: 
  * *The Issue*: Enforcing legacy packages like `numpy<2.0.0` from lecturer slides will fail completely during installation (requires compilation, which fails due to C-API changes). spaCy and PyTorch might also lack pre-compiled binary wheels.
  * *The Solution*: Enforce `numpy>=2.1.0`. If spaCy or PyTorch fail to build from source, you must downgrade your virtual environment to Python 3.12 or 3.13.
* **Python 3.12 & 3.13 (Modern Stable)**:
  * *The Issue*: Normal environment, but legacy slides using NumPy 1.x (`numpy<2.0.0`) can still cause friction with modern packages.
  * *The Solution*: Let pip install the latest compatible NumPy version.
* **Python 3.9 to 3.11 (Classic Stable)**:
  * *The Issue*: Standard environment. Safest version for legacy NumPy 1.x if forced.
* **All Versions (NLTK 3.9+ update)**:
  * *The Issue*: Legacy `punkt` and `averaged_perceptron_tagger` throw a `LookupError` in newer NLTK releases.
  * *The Solution*: Explicitly download `punkt_tab` and `averaged_perceptron_tagger_eng`.

## 📦 1. Auto-Detecting Python Installation

Checks the Python version and installs the correct compatible library stack.

In [ ]:
import sys
import subprocess

py_major = sys.version_info.major
py_minor = sys.version_info.minor
print(f"--- Running Python Version Check: {py_major}.{py_minor} ---")

# 1. Upgrade build tools
print("Step 1: Upgrading pip, setuptools, and wheel...")
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

# 2. Compile package list
packages = [
    "nltk", "spacy", "gradio", "ipywidgets", "matplotlib", 
    "scikit-learn", "sentence_transformers", "transformers", 
    "huggingface_hub", "textblob"
]

# 3. Version checks for NumPy
if py_major == 3 and py_minor >= 14:
    print("\n⚠️ Python 3.14+ detected!")
    print("-> Enforcing modern NumPy 2.x (numpy>=2.1.0) because numpy<2.0.0 will fail to install.")
    print("-> If spaCy or PyTorch installation crashes, you should downgrade your venv to Python 3.12 or 3.13.")
    packages.append("numpy>=2.1.0")
    print("-> Skipping gensim due to compatibility issue (version needs to be <= 3.13)")
    print()
else:
    print("\n✅ Python < 3.14 detected.")
    print("-> Installing standard package configuration.")
    packages.append("numpy") # Let pip resolve the best modern package
    packages.append("gensim")

# 4. Install packages
print(f"Installing: {' '.join(packages)}")
result = subprocess.run([sys.executable, "-m", "pip", "install"] + packages)

if result.returncode == 0:
    print("\n✅ Core libraries successfully installed!")
else:
    print("\n❌ Error encountered during installation. Please review the output logs above.")

## 💾 2. Download NLTK Corpora & Tokenizers

Downloads all datasets, punctuation tables, POS tagger files, and dictionaries. This script covers both legacy and modern NLTK 3.9+ environments (`punkt` / `punkt_tab` and tagger variations).

In [ ]:
import nltk

nltk_resources = [
    "gutenberg",                        # Week 2 (Emma text)
    "punkt",                            # Week 2 (Legacy sentence tokenizer)
    "punkt_tab",                        # Week 2/3 (Modern NLTK 3.9+ sentence tokenizer)
    "wordnet",                          # Week 3 & 10 (Lemmatization, WSD)
    "stopwords",                        # Week 3 (Stopwords list)
    "averaged_perceptron_tagger",       # Week 5 (Legacy NLTK POS Tagger)
    "averaged_perceptron_tagger_eng",   # Week 5 (Modern NLTK 3.9+ English POS Tagger)
    "brown"                             # Week 9 (N-grams)
]

for resource in nltk_resources:
    print(f"Downloading {resource}...")
    try:
        nltk.download(resource)
    except Exception as e:
        print(f"Skipping {resource} (may not be supported on this NLTK version): {e}")

## 🧠 3. Download spaCy English Models

Downloads the small, medium, and large (which contains full pre-trained GloVe embeddings and is ~560MB) English pipeline models.

In [ ]:
import sys
import subprocess

spacy_models = ["en_core_web_sm", "en_core_web_md", "en_core_web_lg"]
for model_name in spacy_models:
    print(f"Downloading {model_name} into the current kernel environment...")
    result = subprocess.run([sys.executable, "-m", "spacy", "download", model_name])
    if result.returncode != 0:
        print(f"⚠️ Failed to download {model_name}. Check the log above.")


## 🤗 4. Pre-Cache Hugging Face Models

Downloads and caches standard Hugging Face transformer models to your local machine (saving hundreds of megabytes in downloads during the exam).

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoModelForCausalLM,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
)
from sentence_transformers import SentenceTransformer

print("=== 1. Downloading Sentence Transformers ===")
st_models = ['all-MiniLM-L12-v2', 'all-MiniLM-L6-v2', 'all-mpnet-base-v2']
for model_name in st_models:
    print(f"Caching {model_name}...")
    SentenceTransformer(model_name)

def cache_auto_model(model_name, model_cls):
    """Cache tokenizer + model without relying on pipeline task aliases."""
    print(f"Caching {model_name}...")
    AutoTokenizer.from_pretrained(model_name)
    model_cls.from_pretrained(model_name)

print("\n=== 2. Downloading Core Hugging Face Models ===")
cache_auto_model("distilbert-base-uncased-finetuned-sst-2-english", AutoModelForSequenceClassification)
cache_auto_model("gpt2", AutoModelForCausalLM)

print("\n=== 3. Downloading Common Exam Model Extensions ===")
# NER (Named Entity Recognition)
cache_auto_model("dslim/bert-base-NER", AutoModelForTokenClassification)
cache_auto_model("dbmdz/bert-large-cased-finetuned-conll03-english", AutoModelForTokenClassification)

# Zero-shot Classification / NLI
cache_auto_model("facebook/bart-large-mnli", AutoModelForSequenceClassification)

# Question Answering
cache_auto_model("distilbert-base-cased-distilled-squad", AutoModelForQuestionAnswering)

# Summarization
cache_auto_model("t5-small", AutoModelForSeq2SeqLM)

print("\nAll models successfully cached!")


# 📂 Revision

## Week 2: Basic Text Operations
### Core Concepts:
1. **Sentence Tokenization (`sent_tokenize`)**: Splits a paragraph into individual sentences by scanning punctuation markers (like periods, exclamation marks, question marks) while handling abbreviations correctly (e.g., "Dr. Smith" stays as one block).
2. **Word Tokenization (`word_tokenize`)**: Splits a sentence into individual tokens (words, punctuation marks, contractions).
3. **Word Frequency Counting (`Counter`)**: Counts occurrences of words. Typically needs normalization (lowercasing, removing non-alpha tokens, filtering short filler words) before plotting.

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from collections import Counter

text = "Natural Language Processing is exciting! Dr. Lubani said it starts at 12:00 PM. Let's do tokenization."

# 1. Tokenize into sentences (Uses punkt_tab behind the scenes in modern NLTK)
sentences = sent_tokenize(text)
print("Sentences:", sentences)

# 2. Tokenize into words
words = word_tokenize(text.lower())
print("\nWords:", words)

# 3. Frequency count of alphanumeric words longer than 2 characters
cleaned_words = [w for w in words if w.isalnum() and len(w) > 2]
word_freq = Counter(cleaned_words)
print("\nTop 3 Word Frequencies:", word_freq.most_common(3))

## Week 3: Text Preprocessing & Gradio Interfaces
### Core Concepts:
1. **Regex Cleaning (`re.sub`)**: Cleans raw text by stripping HTML tags (`<.*?>`), escape characters (`\n`), and punctuation/special symbols.
2. **Stemming (`PorterStemmer`)**: Rule-based chopping of suffixes (e.g., *arguing*, *argued* -> *argu*). Lightweight and fast, but doesn't always yield dictionary words.
3. **Lemmatization (`WordNetLemmatizer` & `spaCy`)**:
   * **NLTK Lemmatizer**: Needs parts-of-speech context (noun, verb, adj) to yield base forms correctly (e.g., *went* -> *go* only if tagged as `'v'`).
   * **spaCy Lemmatizer**: Automatically performs POS tagging internally to lemmatize accurately without manual POS arguments.
4. **Stop Words Removal**: Filtering out extremely frequent but semantic-sparse words (e.g., *the, is, and*).

In [ ]:
import re
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
import spacy

raw_html = "A fair number of upgraded souls <br> went to the library."

# 1. Regex Clean
clean = re.sub(r"<.*?>", " ", raw_html)  # strip HTML tags
clean = re.sub(r"\s+", " ", clean).strip().lower() # normalize spacing
print("Cleaned Text:", clean)

# 2. Stemming vs Lemmatization
porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print("\n[Stemming vs Lemmatization]")
print("Stemming 'upgraded':", porter.stem("upgraded"))
print("Lemmatizing 'went' (no POS):", lemmatizer.lemmatize("went"))
print("Lemmatizing 'went' (verb POS):", lemmatizer.lemmatize("went", 'v'))

# 3. spaCy Pipeline (POS + Lemmatization out-of-the-box)
nlp = spacy.load("en_core_web_md")
doc = nlp(clean)
print("\n[spaCy Auto-Lemmatization]")
for token in doc:
    print(f"{token.text} -> lemma: {token.lemma_}")

# 4. Stopwords filter (combining NLTK & spaCy lists for safety)
nltk_stops = set(stopwords.words('english'))
filtered_words = [token.lemma_ for token in doc if token.lemma_ not in nltk_stops]
print("\nFiltered Lemmas (No Stopwords):", filtered_words)

## Week 5: POS Tagging & Vectorization
### Core Concepts:
1. **Part-of-Speech Tagging (POS)**: Grammatically tagging words (noun, verb, adjective, etc.). NLTK uses the Perceptron Tagger (Penn Treebank format), while spaCy uses statistical neural net tags (both UD universal tags and Penn Treebank fine-grained tags).
2. **Word Embeddings**: Continuous dense vectors representing word meaning (e.g., 300 dimensions in `en_core_web_md` model). Cosine similarity measures closeness of these vectors.
3. **Dimensionality Reduction (PCA)**: Projecting high-dimensional embeddings (e.g., 300D) into a visualizable 2D coordinate plane.

In [ ]:
import nltk
import spacy
from sklearn.decomposition import PCA

sentence = "The quick brown fox jumps over the lazy dog."

# 1. NLTK POS Tagging (Uses averaged_perceptron_tagger_eng in modern NLTK)
tokens = nltk.word_tokenize(sentence)
print("NLTK POS:", nltk.pos_tag(tokens)[:4])

# 2. spaCy POS & Embeddings
nlp = spacy.load("en_core_web_md")
doc = nlp("king queen apple")
print("\n[spaCy POS & Similarity]")
for token in doc:
    print(f"{token.text:<6} POS: {token.pos_:<5} Vector Size: {token.vector.shape}")

print(f"\nSimilarity (king vs queen): {doc[0].similarity(doc[1]):.4f}")
print(f"Similarity (king vs apple): {doc[0].similarity(doc[2]):.4f}")

# 3. PCA Dimensionality Reduction
vectors = [token.vector for token in doc]
pca = PCA(n_components=2)
coords = pca.fit_transform(vectors)
for token, coord in zip(doc, coords):
    print(f"{token.text:<6} projected 2D coordinates: {coord}")

## Week 6: Constituency Parsing (CFG) & Contextual Embeddings
### Core Concepts:
1. **CFG Constituency Parsing**: Grammatical rules forming phrase structural trees. Sentences build upward hierarchically (`S -> NP VP`, `NP -> Det Noun`).
2. **Contextual Sentence Embeddings (`SentenceTransformer`)**: Unlike static embeddings (like Word2Vec/GloVe), context-aware models (BERT-based) capture differences of a word based on neighboring words (e.g., "river bank" vs "savings bank").
3. **Hugging Face Pipelines (`pipeline`)**: High-level API to run pre-trained neural networks for downstream tasks (sentiment classification, NER, generation).

In [ ]:
import nltk
from nltk import CFG
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# 1. Parse a simple CFG Grammar
grammar = CFG.fromstring("""
  S -> NP VP
  NP -> Det Noun
  VP -> Verb NP
  Det -> 'the'
  Noun -> 'dog' | 'cat'
  Verb -> 'chased'
""")
parser = nltk.ChartParser(grammar)
tokens = "the dog chased the cat".split()
print("[Constituency Tree]")
for tree in parser.parse(tokens):
    tree.pretty_print()

# 2. Compute Contextual Similarity
model = SentenceTransformer('all-MiniLM-L12-v2')
sentences = ["A cat sat on the mat.", "A dog lay on the rug."]
embeddings = model.encode(sentences)
sim = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
print(f"SentenceTransformer Cosine Similarity: {sim:.4f}")

# 3. Run Hugging Face Sentiment Analysis Pipeline
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
res = classifier("NLP open book exams are extremely interesting!")[0]
print(f"\nSentiment Label: {res['label']} (Score: {res['score']:.4f})")

## Week 8: CFG Recursion & Dependency Parsing
### Core Concepts:
1. **Prepositional Phrase Recursion**: Nesting preposition structures in grammar definition (e.g. `NP -> NP PP` and `PP -> Prep NP`). This recursion allows parse trees to grow indefinitely to cover complex preposition modifications.
2. **spaCy Dependency Parsing**: Builds grammatical dependency trees linking words via head-modifier relationships. Instead of matching structural phrases, it builds path arrows connecting modifier tokens to their syntactic root verbs/nouns.

In [ ]:
import nltk
from nltk import CFG
import spacy

# 1. Recursive Preposition Structure in CFG
recursive_grammar = CFG.fromstring("""
  S -> NP
  NP -> Det Noun | NP PP
  PP -> Prep NP
  Det -> 'the'
  Noun -> 'key' | 'shelf' | 'door'
  Prep -> 'on' | 'by'
""")
parser = nltk.ChartParser(recursive_grammar)
tokens = "the key on the shelf by the door".split()
trees = list(parser.parse(tokens))
if trees:
    print("[Recursive Parse Tree]")
    trees[0].pretty_print()

# 2. Dependency Parsing in spaCy
nlp = spacy.load("en_core_web_sm")
doc = nlp("The dog chased the cat.")
print("\n[Dependency Head-Modifier Links]")
for token in doc:
    print(f"{token.text:<8} <--{token.dep_:<8}-- head: {token.head.text}")

## Week 9: Statistical Language Modeling & GPT-2
### Core Concepts:
1. **N-Grams**: Extracting contiguous sequences of $N$ items from text. E.g., Bigrams ($N=2$) and Trigrams ($N=3$).
2. **Next Word Prediction**: Estimating the conditional probability of $w_i$ given prior context. For Bigrams, context is $w_{i-1}$; for Trigrams, context is $(w_{i-2}, w_{i-1})$.
3. **Autoregressive Generation (GPT-2)**: Generates text by sequentially predicting and sampling next tokens. Uses neural attention vectors rather than statistical frequency overlap.

### ⚠️ Gensim 4.0+ Library Gotchas:
If your lecturer asks you to perform vector operations using `gensim` (Word2Vec) instead of spaCy, watch out for these massive API breaks between legacy slides and Gensim 4.0+:
* **Vocabulary Access**: Use `model.wv.key_to_index` (Gensim 4+) instead of `model.vocab` (Legacy).
* **Vector Access**: Use `model.wv[word]` (Gensim 4+) instead of `model[word]` (Legacy).
* **Similarity Functions**: Use `model.wv.similarity(w1, w2)` (Gensim 4+) instead of `model.similarity(w1, w2)` (Legacy).

In [ ]:
from nltk.util import ngrams
from collections import Counter
from transformers import pipeline, set_seed

corpus = "the doctor said the patient should rest and the doctor said the patient should take medicine"
tokens = corpus.split()

# 1. Unigram, Bigram, and Trigram structures
bigram_list = list(ngrams(tokens, 2))
trigram_list = list(ngrams(tokens, 3))
print("Sample Bigrams:", bigram_list[:3])
print("Sample Trigrams:", trigram_list[:3])

# 2. Simple Bigram context matching
context = "the"
candidates = [pair[1] for pair in bigram_list if pair[0] == context]
freq = Counter(candidates)
print(f"\nWords following '{context}':", freq.most_common(2))

# 3. GPT-2 Generation
set_seed(42)
generator = pipeline("text-generation", model="gpt2")
prompt = "The doctor said the patient might have to"
res = generator(prompt, max_new_tokens=3, num_return_sequences=1)[0]
print("\nGPT-2 Autoregressive output:", res['generated_text'])

## Week 10: WordNet & Word Sense Disambiguation (WSD)
### Core Concepts:
1. **WordNet Synsets**: Cognitive synonyms grouped into synset records (e.g. `bank.n.01`). Hypernym path routes reveal hierarchical structures.
2. **Path Similarity**: Inverse path length between synsets in the WordNet tree hierarchy: $1 / (PathLength + 1)$. Range: $0.0$ to $1.0$.
3. **Lesk Algorithm (WSD)**: Traditional word disambiguation choosing the word sense definition (gloss) sharing the highest overlap of words with target context sentence.
4. **Gloss Embedding Similarity (WSD)**: Modern neural WSD comparing contextual embeddings of the target sentence against the embeddings of WordNet definitions.

In [ ]:
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk
from sentence_transformers import SentenceTransformer, util

# 1. Synsets and definitions
synsets = wn.synsets('bank', pos=wn.NOUN)
print("Noun senses of 'bank':", [syn.name() for syn in synsets[:3]])
print("First sense definition:", synsets[0].definition())

# 2. Path Similarity between WordNet synsets
dog = wn.synset('dog.n.01')
cat = wn.synset('cat.n.01')
print(f"\nPath Similarity (dog vs cat): {dog.path_similarity(cat):.4f}")

# 3. Traditional Lesk WSD
sentence = "she went to the bank to deposit money"
tokens = sentence.split()
sense = lesk(tokens, 'bank', 'n')
print(f"\nLesk Predicted Sense: {sense.name()} ({sense.definition()})")

# 4. Contextual Gloss WSD using Sentence Transformers
model = SentenceTransformer('all-MiniLM-L6-v2')
sentence_emb = model.encode(sentence, convert_to_tensor=True)
gloss_emb = model.encode(synsets[0].definition(), convert_to_tensor=True)
print(f"Gloss Cosine Similarity: {util.pytorch_cos_sim(sentence_emb, gloss_emb).item():.4f}")
